In [ ]:
#| default_exp training_pipeline
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from tqdm.auto import tqdm
import os
import joblib

# Imports from our previous modules
from adia_cybernet.core_data_processing import ECADataGenerator, SeriesProcessor, PermutationSymbolizer
from adia_cybernet.core_model_architecture import MDL_AU_Net_Autoencoder, StructuralBreakClassifier, HierarchicalArgs

# Introduction
This module contains the high-level orchestration logic for training our model. Our strategy involves a sophisticated two-stage training regimen designed to build a robust "Financial Seismologist":
1. MDL Pre-training (The "School"): We first train our MDL_AU_Net_Autoencoder on a vast, synthetic dataset of Elementary Cellular Automata (ECA) dynamics. The model learns the fundamental "physics" of rule-based systems by performing two tasks simultaneously: reconstructing the system's evolution and classifying the underlying rule from a compressed "fingerprint." This is managed by the MDLPreTrainer class.
1. Fine-tuning (The "On-the-Job Training"): We then take the pre-trained, "educated" encoder and adapt it for the specific task of detecting structural breaks in financial data. We pair it with a new classification head to form a StructuralBreakClassifier. This model is then fine-tuned on the real ADIA training data. This stage is managed by the BreakClassifierFinetuner class.
Finally, the train() function orchestrates this entire process, saving the final, fine-tuned encoder—our ready-to-use artifact for inference.

---

# Section 1: The MDL Pre-Trainer
This class encapsulates all the logic for Stage 1: pre-training our U-Net autoencoder on synthetic ECA data.

In [ ]:
#| export
class MDLPreTrainer:
    """Orchestrates the MDL pre-training stage on synthetic ECA data."""
    def __init__(self, model: MDL_AU_Net_Autoencoder, config: dict):
        self.model = model
        self.config = config
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=config['pretrain_lr'])
        self.recon_criterion = nn.BCEWithLogitsLoss() # Good for binary reconstruction
        self.class_criterion = nn.CrossEntropyLoss()
        self.device = config.get('device', 'cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

    def pretrain(self, data_generator: ECADataGenerator) -> MDL_AU_Net_Autoencoder:
        """Runs the pre-training loop and returns the trained model."""
        print("--- Starting Stage 1: MDL Pre-training ---")
        self.model.train()
        
        # Generate the synthetic dataset
        X, y = data_generator.generate_training_data()
        dataset = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(y).long())
        loader = DataLoader(dataset, batch_size=self.config['pretrain_batch_size'], shuffle=True)
        
        for epoch in range(self.config['pretrain_epochs']):
            epoch_loss = 0
            progress_bar = tqdm(loader, desc=f"Pre-train Epoch {epoch+1}/{self.config['pretrain_epochs']}")
            
            for sequences, labels in progress_bar:
                sequences, labels = sequences.to(self.device), labels.to(self.device)
                
                self.optimizer.zero_grad()
                
                recon, logits = self.model(sequences)
                
                loss_recon = self.recon_criterion(recon, sequences)
                loss_class = self.class_criterion(logits, labels)
                
                # The core MDL dual-loss objective
                total_loss = self.config['alpha_loss'] * loss_recon + \
                             self.config['beta_loss'] * loss_class
                
                total_loss.backward()
                self.optimizer.step()
                
                epoch_loss += total_loss.item()
                progress_bar.set_postfix({'loss': total_loss.item()})

            print(f"Pre-train Epoch {epoch+1} Average Loss: {epoch_loss / len(loader):.4f}")
            
        print("--- MDL Pre-training Complete ---")
        return self.model